In [1]:
!pip install -q langchain
!pip install -q langchain_groq
!pip install -q langchain-community wikipedia
!pip install -q langchain_huggingface
!pip install -q chromadb
%pip install -U langchain
%pip install -q langchain-classic


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.2/577.2 kB 8.5 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.1
    U

In [2]:
from langchain_community.document_loaders import WikipediaLoader
import wikipedia
wikipedia.set_user_agent(
    "MyRAGProject/1.0 (contact: omarahmrednagy122@gmail.com)"
)

/tmp/ipykernel_1202/3261438861.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


In [3]:
query="عمر بن الخطاب"
loader=WikipediaLoader(query=query,
                    load_max_docs=1,
                    doc_content_chars_max=50_000,
                    lang='ar'
                       )
docs=loader.load()
doc=docs[0].page_content

In [4]:
from langchain_text_splitters import NLTKTextSplitter
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
splitter=NLTKTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
metadata=[{"document":query}]
chunks=splitter.create_documents(
    [doc],
    metadatas=metadata
)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
print(len(chunks))
print(chunks[5])

127
page_content='ويجتمع نسبه مع الرسول محمد في كعب بن لؤي بن غالب.

أمه: حنتمة بنت هاشم بن المغيرة بن عبد الله بن عمر بن مخزوم بن يقظة بن كلاب بن مرة بن كعب بن لؤي بن غالب بن فهر بن مالك بن النضر وهو قريش بن كنانة بن خزيمة بن مدركة بن إلياس بن مضر بن نزار بن معد بن عدنان.

وهي ابنة عمّ كلٍ من أم المؤمنين أم سلمة والصحابي خالد بن الوليد وعمرو بن هشام المعروف بلقب أبي جهل.

ويجتمع نسبها مع النبي محمد في كلاب بن مرة.

=== نشأته ===
ولد بعد عام الفيل، وبعد مولد الرسول محمد بثلاث عشرة سنة.' metadata={'document': 'عمر بن الخطاب'}


In [6]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.5,
    max_tokens=128,
    api_key=userdata.get("GROQ_API_KEY"),
)

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (7 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [8]:
from os import path
from langchain_community.vectorstores import Chroma
path="./vector_db"
vector_db=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=path,
)


In [9]:
stuff_template="\n".join([
    "Answer the question using the provided context",
    "if you have no answer return 'بطاطا'"
    "###Context:",
    "{context}",
    "",
    "Question:",
    "{question}",
    "",
])
stuff_prompt = PromptTemplate(
    template=stuff_template,
    input_variables=["context","question"],
    verbose=True

)

In [10]:
question="مولد عمر بن الخطاب؟"
similar_docs=vector_db.similarity_search(question,k=5)


In [11]:
similar_docs[0]

Document(metadata={'document': 'عمر بن الخطاب'}, page_content='== بداية حياته ==\n\n\n=== نسبه ===\n\nهو: عمر بن الخطاب بن نُفَيْل بن عبد العزى بن رياح بن عبد الله بن قرط بن رزاح بن عدي بن كعب بن لؤي بن غالب بن فهر بن مالك بن النضر وهو قريش بن كنانة بن خزيمة بن مدركة بن إلياس بن مضر بن نزار بن معد بن عدنان، العدوي القرشي.\n\nوهو ابن عمّ زيد بن عمرو بن نفيل الموحد على دين إبراهيم.\n\nوأخوه الصحابي زيد بن الخطاب والذي كان قد سبق عمر إلى الإسلام.\n\nويجتمع نسبه مع الرسول محمد في كعب بن لؤي بن غالب.')

In [12]:
print(type(similar_docs))
print(type(similar_docs[0]))
print(similar_docs[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='== بداية حياته ==


=== نسبه ===

هو: عمر بن الخطاب بن نُفَيْل بن عبد العزى بن رياح بن عبد الله بن قرط بن رزاح بن عدي بن كعب بن لؤي بن غالب بن فهر بن مالك بن النضر وهو قريش بن كنانة بن خزيمة بن مدركة بن إلياس بن مضر بن نزار بن معد بن عدنان، العدوي القرشي.

وهو ابن عمّ زيد بن عمرو بن نفيل الموحد على دين إبراهيم.

وأخوه الصحابي زيد بن الخطاب والذي كان قد سبق عمر إلى الإسلام.

ويجتمع نسبه مع الرسول محمد في كعب بن لؤي بن غالب.' metadata={'document': 'عمر بن الخطاب'}


In [13]:
from langchain_classic.chains.question_answering import load_qa_chain

In [14]:
stuff_chain=load_qa_chain(llm,chain_type="stuff",prompt=stuff_prompt)
answer=stuff_chain(
    {
        "input_documents":similar_docs,
        "question":question
     },
    return_only_outputs=True,
)
print (answer)

/tmp/ipykernel_1202/3003134901.py:1: LangChainDeprecationWarning: The function `load_qa_chain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  stuff_chain=load_qa_chain(llm,chain_type="stuff",prompt=stuff_prompt)
/tmp/ipykernel_1202/3003134901.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  answer=stuff_chain(


{'output_text': 'ولد بعد عام الفيل، وبعد مولد الرسول محمد بثلاث عشرة سنة.'}


In [15]:
mr_template="\n".join([
    "Answer the question using only the provided context.",
    "Extract any relevant information that helps answer the question.",
    "Your answer can contain partial information.",
    "If the context contains no relevant information, return exactly 'بطاطا'.",
    "Do not use external knowledge or make unsupported assumptions.",
    "### Context:",
    "{context}",
    "",
    "Question:",
    "{question}",
    "",
])

mr_prompt = PromptTemplate(
    template=mr_template,
    input_variables=["context","question"],
    verbose=True

)

In [16]:
combine_template="\n".join([
    "Given the intermediate answers, combine all relevant information into a final answer.",
    "Ignore any intermediate answer that is exactly 'بطاطا'.",
    "Combine partial information from different answers whenever possible.",
    "If the combined information is insufficient to answer the question, return exactly 'بطاطا'.",
    "Do not use external knowledge or make unsupported assumptions.",
    "Answer in the same language as the question.",
    "### Summaries:",
    "{summaries}",
    "",
    "Question:",
    "{question}",
    "",
    "Final Answer:",
])
combine_prompt = PromptTemplate(
    template=combine_template,
    input_variables=["summaries","question"],
    verbose=True

)


In [17]:
rm_chain=load_qa_chain(
    llm,
    chain_type="map_reduce",
    return_intermediate_steps=True,
    question_prompt=mr_prompt,
    combine_prompt=combine_prompt
    )


In [18]:
question="ما المعارك التي خاضها عمر بن الخطاب؟"
similar_docs=vector_db.similarity_search(question,k=5)

In [19]:
answer=rm_chain({
    "input_documents":similar_docs,
    "question":question
},
                return_only_outputs=True,
                )
print(answer)


/usr/local/lib/python3.13/dist-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'intermediate_steps': ['بطاطا', 'بطاطا', 'غزوة خيبر، غزوة حنين، حصار الطائف، غزوة تبوك.', 'بطاطا', 'بطاطا'], 'output_text': 'غزوة خيبر، غزوة حنين، حصار الطائف، غزوة تبوك.'}


In [20]:
init_template="\n".join([
     "Answer the question using only the provided context.",
    "Extract all relevant information, even if it is partial.",
    "Do not use external knowledge or unsupported assumptions.",
    "If the context contains no relevant information, return exactly 'بطاطا'.",
    "Answer in the same language as the question.",
    "### Context:",
    "{context_str}",
    "",
    "Question:",
    "{question}",
    "",
    "Initial Answer:",
])
init_prompt = PromptTemplate(
    template=init_template,
    input_variables=["context_str","question"],
    verbose=True

)


In [21]:
refine_template = "\n".join([
    "Your task is to refine an existing answer using additional context.",
    "",
    "Original Question:",
    "{question}",
    "",
    "Existing Answer:",
    "{existing_answer}",
    "",
    "Additional Context:",
    "{context_str}",
    "",
    "Instructions:",
    "1. Read the existing answer and the additional context carefully.",
    "2. Add relevant new information from the additional context.",
    "3. Correct the existing answer if the new context provides better evidence.",
    "4. Preserve all relevant information from the existing answer.",
    "5. Do not repeat information unnecessarily.",
    "6. If the existing answer is exactly 'بطاطا', treat it as no answer yet and try to answer using the additional context.",
    "7. If neither the existing answer nor the additional context provides enough information, return exactly 'بطاطا'.",
    "8. Do not use external knowledge or unsupported assumptions.",
    "9. Answer in the same language as the original question.",
    "",
    "Refined Answer:",
])
refine_prompt = PromptTemplate(
    template=refine_template,
    input_variables=["question","existing_answer","context_str"],
    verbose=True

)

In [22]:
refine_chain=load_qa_chain(
    llm,
    chain_type="refine",
    return_intermediate_steps=True,
    question_prompt=init_prompt,
    refine_prompt=refine_prompt
    )

In [23]:
question="ما المعارك التي خاضها عمر بن الخطاب؟"
similar_docs=vector_db.similarity_search(question,k=5)

In [24]:
answer=refine_chain({
    "input_documents":similar_docs,
    "question":question
},
                return_only_outputs=True,
                )
print(answer)


{'intermediate_steps': ['غزوة بدر', 'غزوة بدر', 'غزوة بدر، وغزوة خيبر، وغزوة حنين، وحصار الطائف، وغزوة تبوك، والمشاركة في فتح مكة.', 'غزوة بدر، وغزوة خيبر، وغزوة حنين، وحصار الطائف، وغزوة تبوك، والمشاركة في فتح مكة، إضافةً إلى المعارك التي حدثت في بلاد الشام.', 'غزوة بدر، وغزوة خيبر، وغزوة حنين، وحصار الطائف، وغزوة تبوك، والمشاركة في فتح مكة، إضافةً إلى المعارك التي حدثت في بلاد الشام.'], 'output_text': 'غزوة بدر، وغزوة خيبر، وغزوة حنين، وحصار الطائف، وغزوة تبوك، والمشاركة في فتح مكة، إضافةً إلى المعارك التي حدثت في بلاد الشام.'}


In [25]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define source and destination paths
source_dir = './vector_db'
destination_dir = '/content/drive/MyDrive/vector_db'

# Copy the directory to your Drive
if os.path.exists(source_dir):
    if os.path.exists(destination_dir):
        # If the destination directory already exists, remove it first to avoid conflicts
        shutil.rmtree(destination_dir)
    shutil.copytree(source_dir, destination_dir)
    print(f"Successfully saved vector_db to: {destination_dir}")
else:
    print(f"Error: Source directory {source_dir} not found.")

Mounted at /content/drive
Successfully saved vector_db to: /content/drive/MyDrive/vector_db
